# Day 4B — SQL Feature Engineering

**User Engagement Intelligence**

In a real company, you rarely start with a clean `users.csv`. You start with
**raw event logs** in a database (sessions, clicks, views). This notebook shows
that data-engineering step: SQL `JOIN` + `GROUP BY` aggregations to rebuild
model features from `data/engagement.db`.

## 1. Connect to SQLite and inspect tables

In [1]:
import sys
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from build_features_sql import build_engagement_db, compute_features_from_db

DB_PATH = PROJECT_ROOT / "data" / "engagement.db"
CSV_PATH = PROJECT_ROOT / "data" / "users_with_clusters.csv"

# Build (or rebuild) the DB from the CSV if needed
if not DB_PATH.exists():
    print("Database missing — building from users_with_clusters.csv ...")
    build_engagement_db(CSV_PATH, DB_PATH)
else:
    print(f"Using existing database: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)

# List tables and row counts
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
print("Tables:")
print(tables)

for t in tables["name"]:
    n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t}", conn)["n"][0]
    print(f"  {t}: {n:,} rows")

Using existing database: D:\User Engagement Intelligence\data\engagement.db
Tables:
                name
0  engagement_events
1           sessions
2              users
  engagement_events: 732,925 rows
  sessions: 183,233 rows
  users: 10,000 rows


In [2]:
print("users sample:")
display(pd.read_sql_query("SELECT * FROM users LIMIT 5", conn))

print("sessions sample:")
display(pd.read_sql_query("SELECT * FROM sessions LIMIT 5", conn))

print("engagement_events sample:")
display(pd.read_sql_query("SELECT * FROM engagement_events LIMIT 5", conn))

users sample:


,user_id,age,device_type
0,U00001,22,Android
1,U00002,55,iOS
2,U00003,49,iOS
3,U00004,39,Android
4,U00005,38,Android


sessions sample:


,user_id,session_date,session_duration_minutes
0,U00001,2026-07-23,11.11
1,U00001,2026-06-30,12.86
2,U00001,2026-07-04,11.77
3,U00001,2026-07-10,9.58
4,U00001,2026-07-11,11.61


engagement_events sample:


,user_id,event_type,event_date
0,U00001,view,2026-07-10
1,U00001,view,2026-07-20
2,U00001,view,2026-07-03
3,U00001,view,2026-07-09
4,U00001,view,2026-07-14


## 2. SQL query 1 — sessions per user (`GROUP BY` + aggregates)

**What it computes:** for each user, number of sessions, average duration, and
days since their most recent session (relative to 2026-08-12).

**Why:** these are the classic "recency / frequency / duration" engagement features.

In [3]:
AS_OF = "2026-08-12"  # must match the date used when the DB was generated

q1 = """
SELECT
    user_id,
    COUNT(*) AS sessions_last_30_days,                    -- frequency
    ROUND(AVG(session_duration_minutes), 2) AS avg_session_duration_minutes,
    -- julianday(a) - julianday(b) = day difference between two dates
    CAST(julianday(?) - julianday(MAX(session_date)) AS INTEGER)
        AS days_since_last_login                          -- recency
FROM sessions
GROUP BY user_id
ORDER BY user_id
LIMIT 10
"""

sessions_agg = pd.read_sql_query(q1, conn, params=(AS_OF,))
sessions_agg

,user_id,sessions_last_30_days,avg_session_duration_minutes,days_since_last_login
0,U00001,24,11.30,20
1,U00002,29,12.79,9
2,U00003,23,10.26,25
3,U00004,13,6.53,25
4,U00005,14,11.37,29
5,U00006,39,13.69,4
6,U00007,11,8.40,35
7,U00008,19,4.44,22
8,U00009,6,0.62,44
9,U00010,4,0.81,39


## 3. SQL query 2 — event counts by type (SQL pivot with `SUM(CASE WHEN …)`)

**What it computes:** per-user counts of views, likes, shares, and notification clicks.

**Why:** models need wide feature columns, not a long event log.

In [4]:
q2 = """
SELECT
    user_id,
    SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END)
        AS content_views_last_30_days,
    SUM(CASE WHEN event_type = 'like' THEN 1 ELSE 0 END)
        AS likes_last_30_days,
    SUM(CASE WHEN event_type = 'share' THEN 1 ELSE 0 END)
        AS shares_last_30_days,
    SUM(CASE WHEN event_type = 'notification_click' THEN 1 ELSE 0 END)
        AS notifications_clicked_last_30_days
FROM engagement_events
GROUP BY user_id
ORDER BY user_id
LIMIT 10
"""

events_agg = pd.read_sql_query(q2, conn)
events_agg

,user_id,content_views_last_30_days,likes_last_30_days,shares_last_30_days,notifications_clicked_last_30_days
0,U00001,67,11,2,11
1,U00002,111,27,2,10
2,U00003,62,12,0,7
3,U00004,45,6,0,4
4,U00005,36,2,0,8
5,U00006,134,27,3,15
6,U00007,21,4,0,4
7,U00008,70,21,3,12
8,U00009,8,1,0,4
9,U00010,11,1,0,0


## 4. SQL query 3 — `JOIN` users with session aggregates

**What it computes:** demographic columns from `users` plus session features.
`LEFT JOIN` keeps users who have **zero** sessions (they still appear, with 0s).

In [5]:
q3 = """
SELECT
    u.user_id,
    u.age,
    u.device_type,
    COALESCE(s.sessions_last_30_days, 0) AS sessions_last_30_days,
    ROUND(COALESCE(s.avg_session_duration_minutes, 0), 2)
        AS avg_session_duration_minutes,
    s.days_since_last_login
FROM users u
LEFT JOIN (
    SELECT
        user_id,
        COUNT(*) AS sessions_last_30_days,
        AVG(session_duration_minutes) AS avg_session_duration_minutes,
        CAST(julianday(?) - julianday(MAX(session_date)) AS INTEGER)
            AS days_since_last_login
    FROM sessions
    GROUP BY user_id
) s ON u.user_id = s.user_id
ORDER BY u.user_id
LIMIT 10
"""

user_sessions = pd.read_sql_query(q3, conn, params=(AS_OF,))
user_sessions

,user_id,age,device_type,sessions_last_30_days,avg_session_duration_minutes,days_since_last_login
0,U00001,22,Android,24,11.30,20
1,U00002,55,iOS,29,12.79,9
2,U00003,49,iOS,23,10.26,25
3,U00004,39,Android,13,6.53,25
4,U00005,38,Android,14,11.37,29
5,U00006,59,Android,39,13.69,4
6,U00007,22,iOS,11,8.40,35
7,U00008,51,iOS,19,4.44,22
8,U00009,27,iOS,6,0.62,44
9,U00010,22,Android,4,0.81,39


## 5. SQL query 4 — full feature table (`users` ⋈ sessions ⋈ events)

**What it computes:** the complete engagement feature set in one query —
what you'd hand to a model (or to pandas for merging with labels/clusters).

In [6]:
q4 = """
SELECT
    u.user_id,
    u.age,
    u.device_type,
    COALESCE(s.sessions_last_30_days, 0) AS sessions_last_30_days,
    ROUND(COALESCE(s.avg_session_duration_minutes, 0), 2)
        AS avg_session_duration_minutes,
    s.days_since_last_login,
    COALESCE(e.content_views_last_30_days, 0) AS content_views_last_30_days,
    COALESCE(e.likes_last_30_days, 0) AS likes_last_30_days,
    COALESCE(e.shares_last_30_days, 0) AS shares_last_30_days,
    COALESCE(e.notifications_clicked_last_30_days, 0)
        AS notifications_clicked_last_30_days
FROM users u
LEFT JOIN (
    SELECT
        user_id,
        COUNT(*) AS sessions_last_30_days,
        AVG(session_duration_minutes) AS avg_session_duration_minutes,
        CAST(julianday(?) - julianday(MAX(session_date)) AS INTEGER)
            AS days_since_last_login
    FROM sessions
    GROUP BY user_id
) s ON u.user_id = s.user_id
LEFT JOIN (
    SELECT
        user_id,
        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END)
            AS content_views_last_30_days,
        SUM(CASE WHEN event_type = 'like' THEN 1 ELSE 0 END)
            AS likes_last_30_days,
        SUM(CASE WHEN event_type = 'share' THEN 1 ELSE 0 END)
            AS shares_last_30_days,
        SUM(CASE WHEN event_type = 'notification_click' THEN 1 ELSE 0 END)
            AS notifications_clicked_last_30_days
    FROM engagement_events
    GROUP BY user_id
) e ON u.user_id = e.user_id
ORDER BY u.user_id
"""

sql_features = pd.read_sql_query(q4, conn, params=(AS_OF,))
print(f"SQL feature table: {sql_features.shape}")
sql_features.head()

SQL feature table: (10000, 10)


,user_id,age,device_type,sessions_last_30_days,avg_session_duration_minutes,days_since_last_login,content_views_last_30_days,likes_last_30_days,shares_last_30_days,notifications_clicked_last_30_days
0,U00001,22,Android,24,11.30,20.0,67,11,2,11
1,U00002,55,iOS,29,12.79,9.0,111,27,2,10
2,U00003,49,iOS,23,10.26,25.0,62,12,0,7
3,U00004,39,Android,13,6.53,25.0,45,6,0,4
4,U00005,38,Android,14,11.37,29.0,36,2,0,8


## 6. Sanity check — SQL features vs original CSV

Counts (sessions, views, likes, …) should match **exactly** because we generated
one event/session row per CSV count. Average duration will differ slightly because
each synthetic session gets small random noise around the user's CSV average.

In [7]:
original = pd.read_csv(CSV_PATH)

compare_cols = [
    "sessions_last_30_days",
    "avg_session_duration_minutes",
    "days_since_last_login",
    "content_views_last_30_days",
    "likes_last_30_days",
    "shares_last_30_days",
    "notifications_clicked_last_30_days",
]

# merge SQL features back onto the original CSV on user_id
merged = original[["user_id"] + compare_cols].merge(
    sql_features[["user_id"] + compare_cols],
    on="user_id",
    suffixes=("_csv", "_sql"),
)

print("Exact-match rate for count / recency columns:")
for col in [
    "sessions_last_30_days",
    "days_since_last_login",
    "content_views_last_30_days",
    "likes_last_30_days",
    "shares_last_30_days",
    "notifications_clicked_last_30_days",
]:
    # users with 0 sessions have NULL days_since in SQL — treat carefully
    a = merged[f"{col}_csv"]
    b = merged[f"{col}_sql"]
    # for days_since: only compare users who had >=1 session in CSV
    if col == "days_since_last_login":
        mask = merged["sessions_last_30_days_csv"] > 0
        match_rate = (a[mask] == b[mask]).mean()
    else:
        match_rate = (a == b).mean()
    print(f"  {col:40s}  {match_rate:.1%}")

# Average duration: expect small MAE from synthetic noise, not exact equality
dur_mask = merged["sessions_last_30_days_csv"] > 0
mae = (
    merged.loc[dur_mask, "avg_session_duration_minutes_csv"]
    - merged.loc[dur_mask, "avg_session_duration_minutes_sql"]
).abs().mean()
print(f"\nMean abs error — avg_session_duration_minutes: {mae:.3f} minutes")

merged[
    [
        "user_id",
        "sessions_last_30_days_csv",
        "sessions_last_30_days_sql",
        "avg_session_duration_minutes_csv",
        "avg_session_duration_minutes_sql",
        "likes_last_30_days_csv",
        "likes_last_30_days_sql",
    ]
].head(10)

Exact-match rate for count / recency columns:
  sessions_last_30_days                     100.0%
  days_since_last_login                     100.0%
  content_views_last_30_days                100.0%
  likes_last_30_days                        100.0%
  shares_last_30_days                       100.0%
  notifications_clicked_last_30_days        100.0%

Mean abs error — avg_session_duration_minutes: 0.266 minutes


,user_id,sessions_last_30_days_csv,sessions_last_30_days_sql,avg_session_duration_minutes_csv,avg_session_duration_minutes_sql,likes_last_30_days_csv,likes_last_30_days_sql
0,U00001,24,24,11.0,11.30,11,11
1,U00002,29,29,13.4,12.79,27,27
2,U00003,23,23,10.2,10.26,12,12
3,U00004,13,13,6.5,6.53,6,6
4,U00005,14,14,11.2,11.37,2,2
5,U00006,39,39,14.3,13.69,27,27
6,U00007,11,11,8.1,8.40,4,4
7,U00008,19,19,4.5,4.44,21,21
8,U00009,6,6,0.5,0.62,1,1
9,U00010,4,4,0.5,0.81,1,1


### Why small discrepancies are expected

- **Exact matches:** session counts and event counts should match 100% — we emitted
  one DB row per CSV count.
- **`days_since_last_login`:** matches for users with ≥1 session (we forced the
  newest session onto `today − days_since`). Users with **0 sessions** have no
  session row, so SQL returns `NULL` instead of the CSV value — a realistic gap
  you'd handle with imputation in production.
- **`avg_session_duration_minutes`:** each synthetic session duration is the CSV
  average ± noise, so the SQL `AVG(...)` is *close* but not identical. In a real
  pipeline the CSV average *is* the SQL average — here the noise is only from
  how we simulated raw rows.

## 7. Programmatic path — `compute_features_from_db()`

Same SQL logic packaged in `src/build_features_sql.py` for reuse outside notebooks.

In [8]:
features_fn = compute_features_from_db(DB_PATH)
print(f"compute_features_from_db -> {features_fn.shape}")
display(features_fn.head())

# Confirm it matches the notebook's q4 result
assert features_fn.shape == sql_features.shape
print("Matches notebook query 4 shape: OK")

conn.close()

compute_features_from_db -> (10000, 10)


,user_id,age,device_type,sessions_last_30_days,avg_session_duration_minutes,days_since_last_login,content_views_last_30_days,likes_last_30_days,shares_last_30_days,notifications_clicked_last_30_days
0,U00001,22,Android,24,11.30,20.0,67,11,2,11
1,U00002,55,iOS,29,12.79,9.0,111,27,2,10
2,U00003,49,iOS,23,10.26,25.0,62,12,0,7
3,U00004,39,Android,13,6.53,25.0,45,6,0,4
4,U00005,38,Android,14,11.37,29.0,36,2,0,8


Matches notebook query 4 shape: OK


## Why this matters (Day 4B takeaway)

In a real company, features almost never arrive as a pre-cleaned CSV. They start
as **raw event logs** (session starts, content views, likes, notification clicks)
sitting in a warehouse or operational database. Analysts and ML engineers write
SQL — `JOIN`s, `GROUP BY`s, window functions — to turn those logs into the
per-user aggregates that models consume.

Days 1–3 intentionally started from a tidy CSV so we could focus on EDA,
clustering, and modeling. **This notebook closes the loop:** it shows the data
engineering step that would normally come *first*, and proves we can go from
`engagement.db` → model-ready features with reusable SQL (`compute_features_from_db`).

That end-to-end story (events → SQL features → clusters → churn model → SHAP)
is what makes this portfolio project look like real applied data science.